# 04 — Move 几何与“良好 move”抽样概率

## 为什么需要单独研究 move geometry

当前 complex144 已经说明，sampler 的困难并不只是“变量太多”，而是：

$$
\boxed{\text{局部 bit move 与 OLC 图上的有效拓扑变化不匹配}}
$$

本笔记把 exchange / reconnect / insertion-removal 放进一个统一的图论框架，并记录 2026-09-28 对 complex144 做的 exact reconnect-state audit。

## 1. Fixed-cardinality、degree-correct 状态的几何形状

设：

$$
N=144,\qquad |E_x|=N-1=143,
$$

且每个顶点满足：

$$
d_v^{in}\le1,\qquad d_v^{out}\le1.
$$

这样的有向图每个弱连通分量只能是：

- 一条 directed path；或
- 一个 directed cycle。

设 path component 数为 $P$。每个 path component 比顶点数少 1 条边，而 cycle component 边数等于顶点数，因此：

$$
|E_x|=N-P.
$$

又因为 $|E_x|=N-1$，所以：

$$
\boxed{P=1}.
$$

因此任何 degree-correct fixed-cardinality state 都具有：

$$
\boxed{\text{一条 path} + C\text{ 个 cycles}}
$$

的形式。

Hamilton path 正好对应：

$$
C=0.
$$

## 2. 当前 R2 reconnect 的几何意义

当前 reconnect 是 directed 2-switch：

$$
(a\to b,\;c\to d)
\longleftrightarrow
(a\to d,\;c\to b).
$$

它保持：

$$
|E_x|,
\quad
d_v^{in},
\quad
d_v^{out}.
$$

因此：

- 若两条被删边来自同一条 Hamilton path，通常会把 path 切成  
  $$\text{path}+\text{cycle};$$
- 若一条边来自 path、另一条来自 cycle，可以把 cycle 吸收到 path；
- 若来自两个不同 cycles，可以合并 cycles；
- 若来自同一个 cycle，可以把一个 cycle 改成多个 cycle。

所以从 Hamilton path 出发，一个非平凡 R2 通常不能直接得到另一条 Hamilton path；它需要先经过含 cycle 的中间态，再用第二个 reconnect 合并回来。

## 3. complex144 的 exact R2 状态图结果

我们不再只看若干 sampler trajectory，而是对现有 41 个 static R2 reconnect templates 做了 exact BFS。

状态空间固定为：

- 143 selected edges；
- 与 Hamilton path 相同的 in/out degree profile；
- 允许中间态为 path + cycles。

complex144 的 192 条 exact Hamilton paths 全部被覆盖，但 R2 reconnect 状态图仍然分成：

$$
\boxed{2\text{ 个 disconnected components}}
$$

大小分别：

$$
1152\text{ states},
\qquad
576\text{ states}.
$$

每个 component 恰好包含：

$$
\boxed{96\text{ Hamilton paths}}.
$$

所以此前在“只看 Hamilton paths 的 relocation graph”中观察到的

$$
96+96
$$

并不是因为我们禁止了 cycle 中间态。

即使允许：

$$
\text{Hamilton path}
\to
\text{path+cycles}
\to
\text{Hamilton path},
$$

R2 仍然无法跨过这两个 basin。

## 4. 当前数据具体缺少什么 move：exact 最小 bridge 是 3-edge reconnect

我们进一步对上述两个 R2 connected components 做了穷举距离检查。

最接近的跨 component 两个状态的 Hamming distance 为：

$$
\boxed{6\text{ bits}}.
$$

也就是：

$$
\boxed{3\text{ selected edges removed} + 3\text{ edges added}}.
$$

而且这 6 条差异边组成 **一个单独的 alternating 6-cycle**，不是多个独立差异块。

因此对 complex144 可以得到一个非常具体的结论：

$$
\boxed{\text{R2 不完备；至少需要一个真正的 R3 / 3-edge reconnect 才能直接跨 basin。}}
$$

这不是猜测，而是当前候选图上的 exact state-space 结果。

一个 witness 的局部形式本质类似：

$$
A\to B\to C\to D
$$

与

$$
A\to C\to B\to D
$$

之间同时替换 3 条边。

也就是说，“交换两个相邻局部 segment / reads”在 edge space 中自然是一个 6-bit、3-edge degree-preserving move。

## 5. 为什么用 alternating cycle 来统一描述 reconnect

把每个 read 拆成两个副本：

- $u_{out}$；
- $v_{in}$。

原有 directed edge：

$$
u\to v
$$

对应 bipartite edge：

$$
u_{out}--v_{in}.
$$

固定 in/out degree 的 selected-edge state 就是这个 bipartite candidate graph 上的一个 $b$-matching。

两个具有相同 degree vector 的状态，其 symmetric difference 会分解成 alternating cycles。

因此自然的 degree-preserving move family 是：

$$
\boxed{\text{沿一个 alternating cycle 整体翻转}}
$$

- length 4 alternating cycle：R2 reconnect；
- length 6：R3 reconnect；
- length $2r$：一般 $r$-edge reconnect。

这比“随机挑 $k$ 条边 exchange”更具有图论意义，因为它从构造上保持 in/out degree。

## 6. 与 directed degree-sequence 理论的关系

已知 directed graph realization 的 switch-chain 理论中，2-switch 单独并不总能使状态空间 irreducible；在一般 directed degree sequence 上，加入 directed 3-cycle reorientation 可以恢复连通性（在不额外限制候选边的标准模型中）。

我们的 OLC 图更严格：只有 261 条 candidate arcs，其他边都属于 forbidden edges。因此标准定理不能直接保证 R2+R3 在本数据上一定完全连通。

但它给出很强的设计提示：

$$
\boxed{\text{R2 + R3 是比“继续增加随机 exchange 大小”更自然的下一层 move basis。}}
$$

而 complex144 的 exact minimum bridge 恰好也是 $3+3$ edge difference，与这个方向一致。

## 7. “抽不中好 move”到底有多严重

现有 static R2 template 数：

$$
41.
$$

在 reconnect-intensity benchmark 中，随机从 41 个 templates 抽取时：

$$
P(\text{eligible})\approx18\%\sim21\%.
$$

eligible 后真正被 Metropolis 接受：

$$
P(\text{accept}\mid\text{eligible})
\approx6.6\%\sim8.3\%.
$$

所以总体：

$$
P(\text{accepted reconnect per raw draw})
\approx1.3\%\sim1.7\%.
$$

这解释了为什么单纯把 reconnect 抽取次数放大 2–4 倍收益很有限。

更重要的是：“eligible”仍不等于“拓扑上有帮助”。

## 8. 具体 checkpoint：概率问题与 move-set 问题必须分开

在一个 degree-conflict=1、含 2 个 cycles 的 checkpoint：

- 41 个 static templates；
- 9 个 eligible；
- 其中 6 个能减少 cycle 数。

因此：

$$
P(\text{cycle-reducing}\mid\text{uniform static})
=
\frac6{41}
\approx14.6\%,
$$

而若只在 eligible moves 中均匀抽：

$$
P(\text{cycle-reducing}\mid\text{eligible})
=
\frac69
\approx66.7\%.
$$

这里“优化抽样概率”非常有效。

但另一个 degree-correct、仅剩 1 个 cycle 的 checkpoint 中：

- 8 个 eligible R2；
- **0 个** cycle-reducing R2。

此时：

$$
\boxed{\text{任何重新加权都无济于事，因为 move set 本身没有正确动作。}}
$$

所以必须先判断：

1. 是好 move 存在但难抽中；
2. 还是当前 move family 根本不存在可修复动作。

## 9. 已测试的 guided reconnect

我们比较了：

1. static uniform：从全部 41 templates 抽，ineligible 则 null；
2. eligible-uniform + MH：只从当前 eligible reconnect 中均匀抽；
3. component-biased + MH：对“被删两条 selected edges 来自不同 weak components”的 reconnect 提高 proposal weight。

在 degree-correct 的 path+cycles 状态中，第三类 move 正是能够合并 component、减少 cycle 的几何类型。

小样本 4 seeds 中：

- static uniform：0/4 feasible；
- eligible-uniform MH：0/4；
- component-biased MH：1/4 得到 Hamilton path，best score 1,313,703。

样本太少，不能做成功率结论，但它至少说明：

$$
\boxed{\text{结构化 proposal bias 比单纯增加抽取次数更值得继续。}}
$$

## 10. 如何提高“良好 move”概率且保持正确 MCMC

### 方案 A：只从 eligible moves 采样

若当前 eligible set 大小为 $m(x)$：

$$
q(x\to x')=\frac1{m(x)}.
$$

反向：

$$
q(x'\to x)=\frac1{m(x')}.
$$

因此 acceptance 使用 Hastings ratio：

$$
\alpha
=
\min\left(
1,
e^{-\Delta S}
\frac{m(x)}{m(x')}
\right).
$$

这样可以完全消除 ineligible null draws。

### 方案 B：结构加权 proposal

给每个 eligible move 一个可计算权重：

$$
w_m(x).
$$

例如 cross-component reconnect 权重大于 same-component reconnect。

则：

$$
q(x\to x')
=
\frac{w_m(x)}{\sum_n w_n(x)}.
$$

再精确计算 reverse proposal probability：

$$
\frac{q(x'\to x)}{q(x\to x')}
$$

即可。

所以 sampler **不必拘泥于 proposal symmetry**；只要 Hastings correction 可计算，就可以强烈偏向有意义的 move。

## 11. 推荐的 move hierarchy

当前最自然的 late-phase move 逻辑：

### Degree 尚未修好

主要用：

$$
k_{ex}=1
$$

以及 full-space 下的 defect-aware insertion/removal。

### Degree=0 且存在 cycles

先枚举 R2 eligible moves：

- 若存在 cross-component / cycle-reducing R2：优先抽它；
- 若不存在：启用 R3 alternating-6-cycle move；
- 若 R3 仍不存在：再考虑 R4 或更长 alternating cycle。

### 已经是 Hamilton path

R2 一步通常会制造 cycle，因此若目标是 weighted refinement，更适合：

- R3 path-preserving relocation / local swap；
- 更一般的 $r$-edge alternating-cycle move；
- 或允许 R2 先进入 path+cycle，再第二步退出。

这使 move family 与当前 graph defect 一一对应，而不是统一盲抽。

## 12. Full-space 的统一解释：alternating paths

Fixed-degree reconnect 对应 alternating cycles。

若允许 full-space 中 degree / cardinality 改变，则自然扩展为：

$$
\boxed{\text{alternating paths}}
$$

其端点正好位于 degree deficit / excess 的位置。

此前测试的：

$$
\{a\to b\}
\leftrightarrow
\{a\to c,c\to b\}
$$

可以视为最小的结构化 cardinality-changing repair。

因此后续可以统一设计：

- alternating cycle：保持 degree；
- alternating path：改变局部 degree / cardinality。

这比把 exchange、reconnect、insertion 当成三个互不相关的技巧更适合作为最终算法框架。